# EDA - Gold Reviews

**Source:** `gold.reviews` (PostgreSQL, `marketpulse` database)

> **Memory note:** The full table is ~1.9 GB due to two 384-dim embedding columns.
> This notebook loads **without embeddings** first (~100 MB), with an optional cell to load them separately if needed.

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

engine = create_engine(os.environ["POSTGRES_GOLD_CONN"])

# All columns EXCEPT the heavy embedding arrays
REVIEWS_QUERY = """
SELECT
    review_id, product_id, rating, title, review_text,
    submission_time, last_mod_time,
    is_recommended, helpful_count, not_helpful_count,
    is_featured, is_incentivized, is_staff_review,
    user_location, skin_tone, skin_type, eye_color,
    hair_color, hair_type, hair_concerns, skin_concerns, age_range,
    review_photo_count,
    review_text_clean, review_text_lemmas, review_text_wordcount,
    title_clean, title_lemmas,
    text_quality_score, helpful_ratio, review_age_days,
    is_short_review, embedding_norm_review, embedding_norm_title,
    revision_date
FROM gold.reviews
"""

df = pd.read_sql(REVIEWS_QUERY, engine)
print(f"Loaded {len(df):,} rows | {df.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")
df.head()

Loaded 186,573 rows | 212.6 MB in memory


,review_id,product_id,rating,title,review_text,submission_time,last_mod_time,is_recommended,helpful_count,not_helpful_count,...,review_text_wordcount,title_clean,title_lemmas,text_quality_score,helpful_ratio,review_age_days,is_short_review,embedding_norm_review,embedding_norm_title,revision_date
0,374464811,P517156,3,It’s a pass for me.,I bought the combination of the dark spot brig...,2025-12-28 05:30:57+00:00,2025-12-29 04:30:33+00:00,False,None,0,...,44,s pass,s | pass,0.952,0.0,98,False,1.000293,0.999776,2026-04-05
1,373627850,P517156,5,,"I’m in love with the results , it moisturizer ...",2025-12-19 15:19:17+00:00,2025-12-21 02:00:59+00:00,True,None,0,...,10,,,0.380,0.0,107,False,1.000191,0.000000,2026-04-05
2,359953554,P517156,5,Twice a day ritual must-haves,On week 3 of using these products together and...,2025-09-25 17:45:29+00:00,2025-09-25 18:32:04+00:00,True,None,0,...,26,twice day ritual haves,twice | day | ritual | have,0.658,0.0,192,False,0.999968,0.999841,2026-04-05
3,356899688,P517156,5,,Best thing to have happened to my skin. I boug...,2025-09-10 00:33:01+00:00,2025-09-10 01:00:25+00:00,True,None,0,...,11,,,0.388,0.0,207,False,1.000133,0.000000,2026-04-05
4,356579039,P517156,5,Must try!,This serum has been a game-changer! I’ve been ...,2025-09-05 05:25:56+00:00,2025-10-03 13:36:51+00:00,True,None,0,...,38,try,try,0.904,0.0,212,False,0.999651,0.999746,2026-04-05


In [ ]:
df.info()

In [ ]:
df.describe()

## (Optional) Load embeddings

Only run this if you need the embedding vectors. This will add ~1.8 GB to memory.

In [ ]:
# Load embeddings in chunks to avoid a single massive allocation
CHUNK = 20_000
embeddings_rt, embeddings_ti = [], []

for offset in range(0, len(df), CHUNK):
    chunk_ids = df["review_id"].iloc[offset:offset + CHUNK].tolist()
    placeholders = ",".join(f"'{rid}'" for rid in chunk_ids)
    q = f"SELECT review_id, review_text_embedding, title_embedding FROM gold.reviews WHERE review_id IN ({placeholders})"
    chunk_df = pd.read_sql(q, engine)
    embeddings_rt.append(chunk_df.set_index("review_id")["review_text_embedding"])
    embeddings_ti.append(chunk_df.set_index("review_id")["title_embedding"])
    print(f"  loaded {min(offset + CHUNK, len(df)):,}/{len(df):,}")

emb_rt = pd.concat(embeddings_rt).reindex(df["review_id"])
emb_ti = pd.concat(embeddings_ti).reindex(df["review_id"])

# Convert to numpy matrices (186K x 384)
review_text_emb = np.array(emb_rt.tolist(), dtype=np.float32)
title_emb = np.array(emb_ti.tolist(), dtype=np.float32)

del embeddings_rt, embeddings_ti, emb_rt, emb_ti
print(f"review_text_emb: {review_text_emb.shape}, title_emb: {title_emb.shape}")
print(f"Embedding memory: {(review_text_emb.nbytes + title_emb.nbytes) / 1e6:.0f} MB")